# SQL Analysis of Loan Default Data

This notebook loads the cleaned loan dataset into SQLite and performs SQL-based analysis for several business-relevant metrics.

In [ ]:
import sqlite3
import pandas as pd

csv_path = '../outputs/cleaned_loan_default.csv'
conn = sqlite3.connect(':memory:')

df = pd.read_csv(csv_path)
df.to_sql('loan_data', conn, index=False, if_exists='replace')

print('Loaded rows:', len(df))
print('Columns:', list(df.columns))

In [ ]:
query_total = '''
SELECT
    COUNT(*) AS total_loans,
    ROUND(SUM(LoanAmount), 2) AS total_loan_amount
FROM loan_data;
'''

pd.read_sql_query(query_total, conn)

In [ ]:
query_defaults = '''
SELECT
    SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS default_count,
    ROUND(AVG(CASE WHEN "Default" = 1 THEN 1.0 ELSE 0.0 END), 4) AS default_rate
FROM loan_data;
'''

pd.read_sql_query(query_defaults, conn)

In [ ]:
query_income_by_default = '''
SELECT
    "Default",
    ROUND(AVG(Income), 2) AS avg_income
FROM loan_data
GROUP BY "Default"
ORDER BY "Default";
'''

pd.read_sql_query(query_income_by_default, conn)

In [ ]:
query_loan_amount_by_default = '''
SELECT
    "Default",
    ROUND(AVG(LoanAmount), 2) AS avg_loan_amount
FROM loan_data
GROUP BY "Default"
ORDER BY "Default";
'''

pd.read_sql_query(query_loan_amount_by_default, conn)

In [ ]:
query_default_rate_by_employment = '''
SELECT
    EmploymentType,
    COUNT(*) AS total_loans,
    ROUND(SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loan_data
GROUP BY EmploymentType
ORDER BY default_rate_pct DESC;
'''

pd.read_sql_query(query_default_rate_by_employment, conn)

In [ ]:
query_default_rate_by_purpose = '''
SELECT
    LoanPurpose,
    COUNT(*) AS total_loans,
    ROUND(SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loan_data
GROUP BY LoanPurpose
ORDER BY default_rate_pct DESC;
'''

pd.read_sql_query(query_default_rate_by_purpose, conn)

In [ ]:
query_default_rate_by_credit_group = '''
SELECT
    CASE
        WHEN CreditScore < 550 THEN 'Below 550'
        WHEN CreditScore < 650 THEN '550-649'
        WHEN CreditScore < 750 THEN '650-749'
        ELSE '750+'
    END AS credit_score_group,
    COUNT(*) AS total_loans,
    ROUND(SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loan_data
GROUP BY
    CASE
        WHEN CreditScore < 550 THEN 'Below 550'
        WHEN CreditScore < 650 THEN '550-649'
        WHEN CreditScore < 750 THEN '650-749'
        ELSE '750+'
    END
ORDER BY
    CASE credit_score_group
        WHEN 'Below 550' THEN 1
        WHEN '550-649' THEN 2
        WHEN '650-749' THEN 3
        WHEN '750+' THEN 4
    END;
'''

pd.read_sql_query(query_default_rate_by_credit_group, conn)

## SQL Summary
This notebook creates a SQLite table named `loan_data` from the cleaned CSV and answers the key credit-risk questions with SQL aggregations.